# 03 — Preprocessing NISAR GCOV Data

Calibration (linear ↔ dB conversion), speckle filtering (Lee filter),
and spatial multilooking on NISAR GCOV backscatter data.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bullocke/nice-sar/blob/main/notebooks/03_preprocessing.ipynb)

## 1. Install and Import

In [ ]:
# Pin fsspec/s3fs to avoid Colab conflicts, then force-reinstall only nice-sar
%pip install -q "fsspec<=2025.3.0" "s3fs<=2025.3.0"
%pip install -q --force-reinstall --no-deps "git+https://github.com/bullocke/nice-sar.git@main"

## Colab Runtime Note

If you already imported `nice_sar` earlier in this Colab runtime, restart the runtime after running the install cell, then run the notebook from the top. Colab can keep the previously imported package in memory even after `%pip install --upgrade`.

In [ ]:
%matplotlib inline

import logging
import os

import matplotlib.pyplot as plt
import numpy as np

from nice_sar.auth.earthdata import (
    get_granule_url,
    get_https_filesystem,
    get_s3_filesystem,
    login,
)
from nice_sar.io.hdf5 import open_nisar
from nice_sar.io.products import read_gcov
from nice_sar.preprocess.calibration import linear_to_db, db_to_linear
from nice_sar.preprocess.filters import lee_filter, refined_lee_filter
from nice_sar.preprocess.multilook import multilook
from nice_sar.viz.display import percentile_stretch

logging.basicConfig(level=logging.INFO, format="%(name)s — %(message)s", force=True)
logger = logging.getLogger(__name__)

# ── Study Areas ──────────────────────────────────────────────────────────────
STUDY_AREAS = {
    "salt_lake_city": (-112.1, 40.5, -111.7, 40.9),
    "cascades_wa": (-122.5, 46.0, -121.5, 47.0),
    "amazon": (-55.0, -3.5, -54.0, -2.5),
}
STUDY_AREA = "salt_lake_city"  # ← change to try other regions
bbox = STUDY_AREAS[STUDY_AREA]
logger.info("Study area: %s  bbox=%s", STUDY_AREA, bbox)


## 2. Load GCOV Data

In [ ]:
try:
    login()
except Exception as e:
    logger.error(
        "Authentication failed: %s\n"
        "Ensure credentials are configured via one of:\n"
        "  1. Environment variables: EARTHDATA_USERNAME / EARTHDATA_PASSWORD\n"
        "  2. ~/.netrc with: machine urs.earthdata.nasa.gov login <user> password <pass>\n"
        "  3. Interactive prompt (Colab / Jupyter only)",
        e,
    )
    raise

if os.environ.get("AWS_DEFAULT_REGION") == "us-west-2":
    fs = get_s3_filesystem()
    granule_access = "s3"
else:
    fs = get_https_filesystem()
    granule_access = "https"

# Search for a granule
import earthaccess

results = earthaccess.search_data(
    short_name="NISAR_L2_GCOV_BETA_V1",
    bounding_box=bbox,
    count=5,
)
granule_url = get_granule_url(results[0], access=granule_access)
logger.info("Using %s granule URL: %s", granule_access.upper(), granule_url)

# Read HH and HV
h5 = open_nisar(granule_url, filesystem=fs)
da_hh = read_gcov(h5, frequency="A", polarization="HH")
da_hv = read_gcov(h5, frequency="A", polarization="HV")
h5.close()

# Use a center subset for fast processing
hh_linear = da_hh.values
hv_linear = da_hv.values
cy, cx = hh_linear.shape[0] // 2, hh_linear.shape[1] // 2
sz = 512
hh = hh_linear[cy - sz : cy + sz, cx - sz : cx + sz]
hv = hv_linear[cy - sz : cy + sz, cx - sz : cx + sz]

logger.info("Subset shape: %s", hh.shape)

## 3. Calibration — Linear ↔ dB Conversion

GCOV diagonal terms are stored as linear power (γ⁰). Convert to dB for
visualization and analysis:

$$\sigma_{dB} = 10 \cdot \log_{10}(\sigma_{linear})$$

In [ ]:
# Convert to dB
hh_db = linear_to_db(hh)
hv_db = linear_to_db(hv)

# Verify round-trip
hh_back = db_to_linear(hh_db)
valid = np.isfinite(hh) & (hh > 0)
max_err = np.abs(hh[valid] - hh_back[valid]).max()
logger.info("dB round-trip max error: %.2e", max_err)

# Compare histograms
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
valid_lin = hh[np.isfinite(hh) & (hh > 0)]
valid_db = hh_db[np.isfinite(hh_db)]

axes[0].hist(valid_lin.ravel(), bins=100, color="steelblue", edgecolor="none")
axes[0].set_title("HH — Linear Power")
axes[0].set_xlabel("γ⁰ (linear)")

axes[1].hist(valid_db.ravel(), bins=100, color="coral", edgecolor="none")
axes[1].set_title("HH — Decibels")
axes[1].set_xlabel("γ⁰ (dB)")

plt.tight_layout()
plt.show()

## 4. Speckle Filtering — Lee Filter

SAR images contain multiplicative speckle noise. The Lee filter reduces speckle
while preserving edges by adapting to local statistics.

In [ ]:
# Apply Lee filter with different window sizes
hh_lee3 = lee_filter(hh, window_size=3)
hh_lee5 = lee_filter(hh, window_size=5)
hh_lee7 = lee_filter(hh, window_size=7)

# Convert to dB for display
images = [hh_db, linear_to_db(hh_lee3), linear_to_db(hh_lee5), linear_to_db(hh_lee7)]
titles = ["Original", "Lee 3×3", "Lee 5×5", "Lee 7×7"]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, img, title in zip(axes, images, titles):
    stretched = percentile_stretch(img, p_low=2, p_high=98)
    ax.imshow(stretched, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")

plt.suptitle("Effect of Lee Filter Window Size on HH Backscatter", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Refined Lee Filter

The refined Lee filter uses directional edge masks to better preserve
linear features while still reducing speckle.

In [ ]:
hh_refined = refined_lee_filter(hh, win=7)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, title in zip(
    axes,
    [hh_db, linear_to_db(hh_lee7), linear_to_db(hh_refined)],
    ["Original", "Lee 7×7", "Refined Lee 7×7"],
):
    stretched = percentile_stretch(img, p_low=2, p_high=98)
    ax.imshow(stretched, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")

plt.suptitle("Lee vs Refined Lee Filter", fontsize=13)
plt.tight_layout()
plt.show()

## 6. Spatial Multilooking

Multilooking averages adjacent pixels to reduce speckle at the cost of
spatial resolution. Common for SAR analysis workflows.

In [ ]:
# Multilook with different factors
hh_ml2 = multilook(hh, looks_y=2, looks_x=2)
hh_ml4 = multilook(hh, looks_y=4, looks_x=4)
hh_ml8 = multilook(hh, looks_y=8, looks_x=8)

logger.info("Original: %s", hh.shape)
logger.info("2×2 multilook: %s", hh_ml2.shape)
logger.info("4×4 multilook: %s", hh_ml4.shape)
logger.info("8×8 multilook: %s", hh_ml8.shape)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, img, title in zip(
    axes,
    [hh_db, linear_to_db(hh_ml2), linear_to_db(hh_ml4), linear_to_db(hh_ml8)],
    [f"Original ({hh.shape[0]}×{hh.shape[1]})",
     f"2×2 ({hh_ml2.shape[0]}×{hh_ml2.shape[1]})",
     f"4×4 ({hh_ml4.shape[0]}×{hh_ml4.shape[1]})",
     f"8×8 ({hh_ml8.shape[0]}×{hh_ml8.shape[1]})"],
):
    stretched = percentile_stretch(img, p_low=2, p_high=98)
    ax.imshow(stretched, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")

plt.suptitle("Effect of Multilooking on Spatial Resolution", fontsize=13)
plt.tight_layout()
plt.show()

## 7. Combined Workflow — Filter then Multilook

A common preprocessing pipeline: filter speckle first, then multilook to
the desired resolution.

In [ ]:
# Pipeline: Lee 5×5 → 2×2 multilook
hh_filtered_ml = multilook(lee_filter(hh, window_size=5), looks_y=2, looks_x=2)
hv_filtered_ml = multilook(lee_filter(hv, window_size=5), looks_y=2, looks_x=2)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original
axes[0].imshow(percentile_stretch(hh_db), cmap="gray", vmin=0, vmax=1)
axes[0].set_title(f"Original ({hh.shape[0]}×{hh.shape[1]})")
axes[0].axis("off")

# Just multilook
axes[1].imshow(percentile_stretch(linear_to_db(hh_ml2)), cmap="gray", vmin=0, vmax=1)
axes[1].set_title(f"2×2 Multilook ({hh_ml2.shape[0]}×{hh_ml2.shape[1]})")
axes[1].axis("off")

# Filter + multilook
axes[2].imshow(percentile_stretch(linear_to_db(hh_filtered_ml)), cmap="gray", vmin=0, vmax=1)
axes[2].set_title(f"Lee 5×5 + 2×2 ML ({hh_filtered_ml.shape[0]}×{hh_filtered_ml.shape[1]})")
axes[2].axis("off")

plt.suptitle("Comparison of Preprocessing Approaches", fontsize=13)
plt.tight_layout()
plt.show()

logger.info("Preprocessing notebook complete.")